In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test_Iceberg_Query").getOrCreate()

print("=== DANH SÁCH CÁC BẢNG TRONG TIKI ===")
spark.sql("SHOW TABLES IN local_catalog.tiki_bronze").show(truncate=False)


=== DANH SÁCH CÁC BẢNG TRONG TIKI ===
+-----------+------------+-----------+
|namespace  |tableName   |isTemporary|
+-----------+------------+-----------+
|tiki_bronze|products_raw|false      |
+-----------+------------+-----------+



In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test_Iceberg_Query").getOrCreate()

print("=== DANH SÁCH CÁC BẢNG TRONG TIKI ===")
spark.sql("SHOW TABLES IN local_catalog.tiki").show(truncate=False)


=== DANH SÁCH CÁC BẢNG TRONG TIKI ===
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|tiki     |price_history|false      |
|tiki     |products     |false      |
+---------+-------------+-----------+



In [20]:
spark.sql("select * from local_catalog.tiki_bronze.products_raw").show()

+---------+-------------+--------------------+--------------------+------+--------------+--------+-------------+--------------------+--------------+------------+--------------------+-----------+-------------+----------+--------------------+--------------------+
|       id|          sku|                name|             url_key| price|original_price|discount|discount_rate|          brand_name|rating_average|review_count|       thumbnail_url|category_id|quantity_sold|crawl_date|           loaded_at|         source_file|
+---------+-------------+--------------------+--------------------+------+--------------+--------+-------------+--------------------+--------------+------------+--------------------+-----------+-------------+----------+--------------------+--------------------+
|279093481|1822096185602|Mặt Nạ Vàng 24k C...|mat-na-vang-24k-c...|460000|        460000|       0|            0|              Karmel|           0.0|           0|https://salt.tiki...|      11709|            0|2026-0

In [23]:
spark.sql("select * from local_catalog.tiki_bronze.products_raw where id = 248681254 order by crawl_date").show(truncate=False)

+---------+-------------+------------------------------+---------------------------------------+------+--------------+--------+-------------+------------+--------------+------------+-----------------------------------------------------------------------------------------------+-----------+-------------+----------+--------------------------+--------------------------------------+
|id       |sku          |name                          |url_key                                |price |original_price|discount|discount_rate|brand_name  |rating_average|review_count|thumbnail_url                                                                                  |category_id|quantity_sold|crawl_date|loaded_at                 |source_file                           |
+---------+-------------+------------------------------+---------------------------------------+------+--------------+--------+-------------+------------+--------------+------------+------------------------------------------------------

In [ ]:
query = """
        select brand_name, round(avg(price), 2)
        from local_catalog.tiki.products
        group by brand_name
"""
spark.sql(query).show()

In [ ]:
query = """
        select count(distinct brand_name) as count_brand
        from local_catalog.tiki.products
"""
spark.sql(query).show()

In [7]:
query = """
        select count(*) as count_product_of_durex
        from local_catalog.tiki.products
        where brand_name = 'Durex'
"""
spark.sql(query).show()

+----------------------+
|count_product_of_durex|
+----------------------+
|                   110|
+----------------------+



In [3]:
print("=== DANH SÁCH DATABASE ===")
spark.sql("SHOW DATABASES IN local_catalog").show(truncate=False)

=== DANH SÁCH DATABASE ===
+-----------+
|namespace  |
+-----------+
|default    |
|tiki       |
|tiki_bronze|
+-----------+



In [ ]:
print("=== DANH SÁCH CÁC BẢNG TRONG TIKI ===")
spark.sql("SHOW TABLES IN local_catalog.tiki").show(truncate=False)

In [ ]:
print("=== CẤU TRÚC BẢNG PRODUCTS ===")
spark.sql("DESCRIBE local_catalog.tiki.products").show(truncate=False)

In [6]:
print("=== LỊCH SỬ CÁC LẦN GHI DỮ LIỆU ===")
spark.sql("SELECT * FROM local_catalog.tiki.products.history").show(truncate=False)

print("=== CHI TIẾT SNAPSHOTS ===")
spark.sql("SELECT committed_at, snapshot_id, operation FROM local_catalog.tiki.products.snapshots").show(truncate=False)

=== LỊCH SỬ CÁC LẦN GHI DỮ LIỆU ===
+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-25 07:41:15.762|212357408956329652 |NULL               |true               |
|2026-05-25 07:42:18.757|8761820206006598383|212357408956329652 |true               |
|2026-05-25 08:49:59.182|4698979343288364922|8761820206006598383|true               |
|2026-05-27 12:28:40.798|8241903772937420669|4698979343288364922|true               |
|2026-05-27 12:31:11.661|6755189955897460993|8241903772937420669|true               |
|2026-05-27 12:54:51.992|8407684545818752076|6755189955897460993|true               |
|2026-05-28 09:06:49.559|9135974879012586705|8407684545818752076|true               |
+-----------------------+-------------------+-------------------+-------------------+

=== CHI TIẾT SNAP

In [9]:
spark.sql("""
    SELECT count(id) as so_sp_bien_dong, crawl_date 
    FROM local_catalog.tiki.price_history 
    GROUP BY crawl_date
""").show()

+---------------+----------+
|so_sp_bien_dong|crawl_date|
+---------------+----------+
|            183|2026-05-28|
|          21027|2026-05-25|
|           1345|2026-05-27|
|           3610|2026-06-02|
+---------------+----------+



In [16]:
spark.sql("""
    select a.id, b.name, b.price as new_price, a.price as old_price, a.crawl_date, b.crawl_date
    from local_catalog.tiki.price_history a
    join local_catalog.tiki.products b
    on a.id = b.id
    where a.price != b.price
""").show(100)

+---------+--------------------+---------+---------+----------+----------+
|       id|                name|new_price|old_price|crawl_date|crawl_date|
+---------+--------------------+---------+---------+----------+----------+
|   505645|Dầu Gội Dưỡng Tóc...|   149000|   134000|2026-05-25|2026-06-02|
|   525402|Sữa Tắm Romano Fo...|   123000|    98000|2026-05-25|2026-06-02|
|   580690|Lăn Khử Mùi Cao C...|    50000|    64000|2026-05-25|2026-06-02|
|   580785|Lăn Khử Mùi Trắng...|    77000|    62000|2026-05-25|2026-06-02|
|   631508|Sữa rửa mặt ngăn ...|    70000|    67000|2026-05-25|2026-06-02|
|   631508|Sữa rửa mặt ngăn ...|    70000|    67000|2026-05-27|2026-06-02|
|   631519|Serum chấm mụn ch...|    94000|    89000|2026-05-25|2026-06-02|
|   767101|Bao cao su Durex ...|   123500|   117000|2026-05-25|2026-06-02|
|  1058862|Tinh Dầu Gừng Biy...|   626000|   569000|2026-05-25|2026-06-02|
|  1058890|Tinh Dầu Tràm Biy...|   141000|   128000|2026-05-25|2026-06-02|
|  1058898|Tinh Dầu Hỗn H

In [ ]:
spark.sql("""
    select * from local_catalog.tiki.price_history 
    where crawl_date = '2026-05-14'
""").show()

In [ ]:
spark.sql("""
    SELECT count(*) as so_luong_sp_moi 
    FROM local_catalog.tiki.price_history 
""").show()

In [ ]:
spark.sql("""
    SELECT count(*) as so_luong_sp_moi 
    FROM local_catalog.tiki.price_history 
""").show()

In [ ]:
spark.sql("""
    SELECT count(*) as so_luong_sp
    FROM local_catalog.tiki.products
""").show()

In [ ]:
spark.sql("""
    SELECT 
        a.id, 
        b.name, 
        a.price as gia_ghi_nhan_cdc,
        b.price as gia_goc
    FROM local_catalog.tiki.price_history a
    JOIN local_catalog.tiki.products b ON a.id = b.id
    WHERE a.crawl_date = '2026-05-15'
""").show(50, truncate=False)